# HWD Susenas — MCAR 10% + Structural Violation Rate (SVR)

Notebook Kaggle untuk menguji HWD pada data Susenas dengan:
- Mekanisme missing **MCAR 10%**
- Evaluasi akurasi imputasi (MAE, RMSE, CRPS)
- **Structural Violation Rate (SVR)** untuk dua skenario:
  1. **Tanpa** survey rule engine — model mengisi semua posisi kosong
  2. **Dengan** survey rule engine — posisi structural dijaga kosong

Model dilatih **sekali**; kedua skenario dibedakan hanya pada pascaproses.

## Cell 1 — Install Dependencies

In [ ]:
!pip install PyWavelets properscoring --quiet

## Cell 2 — Setup Path & File HWD (Kaggle)

In [ ]:
import os, sys, shutil, glob

INPUT_DIR = '/kaggle/input/datasets/cloudyamontolalu/hwdtest'   # sesuaikan dgn dataset Kaggle Anda
WORK_DIR  = '/kaggle/working'
DATA_DIR  = f'{WORK_DIR}/Data'
CKPT_DIR  = f'{WORK_DIR}/Checkpoints'
HWD_DIR   = f'{WORK_DIR}/HWD'

for d in [DATA_DIR, CKPT_DIR, f'{DATA_DIR}/mask/susenas', HWD_DIR]:
    os.makedirs(d, exist_ok=True)

# Salin seluruh berkas kode (.py) + folder models/layers dari input ke HWD_DIR
for pat in ['*.py']:
    for f in glob.glob(f'{INPUT_DIR}/**/{pat}', recursive=True):
        shutil.copy2(f, HWD_DIR)
for sub in ['models', 'layers']:
    for d in glob.glob(f'{INPUT_DIR}/**/{sub}', recursive=True):
        shutil.copytree(d, f'{HWD_DIR}/{sub}', dirs_exist_ok=True)
        break

sys.path.insert(0, HWD_DIR)
os.chdir(HWD_DIR)
print('Isi HWD_DIR:', sorted(os.listdir(HWD_DIR)))

## Cell 3 — Import

In [ ]:
import numpy as np
import pandas as pd
import torch
import time, importlib.util
import properscoring as ps
warnings_shown = False
np.random.seed(1); torch.manual_seed(1)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# Muat survey_rules.py
spec = importlib.util.spec_from_file_location('sr', f'{HWD_DIR}/survey_rules.py')
sr = importlib.util.module_from_spec(spec); spec.loader.exec_module(sr)
print('survey_rules dimuat:', len(sr.SUSENAS_SKIP_RULES), 'rule')

## Cell 4 — Preprocessing Data Susenas

Data Susenas (sudah tergabung antar tahun via kode_sampel) diproses:
- Kolom numerik yang masuk model: R101, R102, R105, R301, R705, R706, R707, food, nonfood, expend, kapita
- kode_sampel, tahun, dan R703_A DIKELUARKAN dari fitur model
- R703_A tetap disimpan terpisah karena dibutuhkan survey rule engine
- Normalisasi Z-score per kolom; nilai hilang ditandai sentinel -200

In [ ]:
# Upload data Susenas
try:
    from google.colab import files
    up = files.upload(); FSUS = list(up.keys())[0]
except Exception:
    # Kaggle: cari file susenas di input
    cands = glob.glob(f'{INPUT_DIR}/**/*usenas*.xlsx', recursive=True) + \
            glob.glob(f'{INPUT_DIR}/**/*usenas*.csv', recursive=True)
    FSUS = cands[0] if cands else 'susenas.xlsx'

df_raw = pd.read_excel(FSUS) if FSUS.lower().endswith(('.xlsx','.xls')) else pd.read_csv(FSUS)
print('Data mentah:', df_raw.shape, '| kolom:', list(df_raw.columns))

# Kolom fitur model (numerik) dan kolom pendukung
FEAT_COLS = ['R101','R102','R105','R301','R705','R706','R707','food','nonfood','expend','kapita']
# samakan nama (jaga-jaga huruf besar/kecil)
colmap = {c.lower(): c for c in df_raw.columns}
def col(name):
    return colmap.get(name.lower(), name)
FEAT_COLS = [col(c) for c in FEAT_COLS]

# Simpan R703_A (untuk rule engine) & data asli sebelum normalisasi
R703_COL = col('R703_A')
df_support = df_raw[[R703_COL] + FEAT_COLS].copy()

# Matriks fitur numerik
X = df_raw[FEAT_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
n_rows, n_feat = X.shape
print(f'Fitur model: {n_feat} kolom, {n_rows} baris')

# Normalisasi Z-score per kolom (dari nilai teramati), simpan mean/std utk denormalisasi
means = np.nanmean(X, axis=0)
stds  = np.nanstd(X, axis=0); stds[stds==0] = 1.0
Xn = (X - means) / stds
Xn[np.isnan(Xn)] = -200.0    # sentinel utk natural missing
print('Normalisasi selesai. Sentinel -200 pada natural missing:', int((Xn==-200).sum()))

## Cell 5 — Windowing & Mask MCAR 10%

Data disusun menjadi window sepanjang seq_len. Baris diperlakukan independen (tanpa
asumsi keterkaitan temporal antarbaris); windowing hanya menata data agar sesuai bentuk
masukan model. Mask MCAR menyembunyikan 10% posisi teramati secara acak seragam sebagai
artificial missing untuk evaluasi.

In [ ]:
SEQ_LEN = 48
MISSING_RATE = 0.1
SEED = 1
rng = np.random.default_rng(SEED)

# Potong agar habis dibagi seq_len, lalu reshape [N, seq_len, n_feat]
n_ok = (n_rows // SEQ_LEN) * SEQ_LEN
data = Xn[:n_ok].reshape(-1, SEQ_LEN, n_feat)
print('Bentuk data:', data.shape)

# Ground-truth observed mask: 1 jika teramati (bukan sentinel), 0 jika natural missing
gt_mask = (data != -200).astype(np.float32)

# Mask MCAR: dari posisi teramati, sembunyikan 10% (0 = disembunyikan/target)
cond_mask = gt_mask.copy()
obs_idx = np.argwhere(gt_mask == 1)
n_hide = int(len(obs_idx) * MISSING_RATE)
hide = obs_idx[rng.choice(len(obs_idx), n_hide, replace=False)]
for (a,b,c) in hide:
    cond_mask[a,b,c] = 0.0
print(f'Artificial missing (MCAR 10%): {n_hide} posisi disembunyikan')
# posisi target evaluasi = gt_mask==1 & cond_mask==0
eval_target = ((gt_mask==1) & (cond_mask==0))
print('Posisi target evaluasi:', int(eval_target.sum()))

## Cell 6 — Konfigurasi Model HWD

In [ ]:
from types import SimpleNamespace
cfg = SimpleNamespace(
    seq_len=SEQ_LEN, enc_in=n_feat, c_out=n_feat,
    d_model=128, e_layers=4, nheads=8, channel=128, proj_t=128,
    residual_layers=4, timeemb=128, featureemb=16,
    diffusion_step_num=50, schedule='quad', beta_start=1e-4, beta_end=0.2,
    epoch_diff=400, learning_rate_diff=1e-3,
    mask_ratio_ssl=0.2, avg_mask_len_ssl=3,
    wavelet='db4', levels=3, batch=16,
    device=DEVICE, missing_rate=MISSING_RATE, seed=SEED, n_samples=100,
)
print('Config siap. enc_in =', cfg.enc_in)

## Cell 7 — Dataset Loader (inline)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class SusenasDataset(Dataset):
    def __init__(self, data, cond_mask, gt_mask):
        self.data = torch.from_numpy(data).float()
        self.cond_mask = torch.from_numpy(cond_mask).float()
        self.gt_mask = torch.from_numpy(gt_mask).float()
        self.tp = torch.arange(data.shape[1], dtype=torch.float32)
    def __len__(self): return self.data.shape[0]
    def __getitem__(self, i):
        return (self.data[i], self.cond_mask[i], self.tp, self.gt_mask[i])

ds = SusenasDataset(data, cond_mask, gt_mask)
train_loader = DataLoader(ds, batch_size=cfg.batch, shuffle=True)
test_loader  = DataLoader(ds, batch_size=cfg.batch, shuffle=False)
print('Dataset:', len(ds), 'window')

## Cell 8 — Training HWD (sekali)

Model dilatih sekali. Selama pelatihan, fungsi loss hanya dihitung pada posisi target
(cond_mask==0 & gt_mask==1), sehingga model belajar merekonstruksi nilai yang disembunyikan.

In [ ]:
from models import main_model
from torch import optim

model = main_model.HWD(cfg).to(cfg.device)
optimizer = optim.Adam(model.parameters(), lr=cfg.learning_rate_diff, weight_decay=1e-6)
p1, p2 = int(0.75*cfg.epoch_diff), int(0.90*cfg.epoch_diff)
sched = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[p1,p2], gamma=0.1)

losses = []
for epoch in range(cfg.epoch_diff):
    model.train(); ep=[]; t0=time.time()
    for obs_d, obs_m, obs_tp, gt_m in train_loader:
        obs_d=obs_d.to(cfg.device); obs_m=obs_m.to(cfg.device); gt_m=gt_m.to(cfg.device)
        optimizer.zero_grad()
        loss = model(obs_d, obs_m, obs_tp.to(cfg.device), gt_m)
        loss.backward(); optimizer.step(); ep.append(loss.item())
    sched.step(); losses.append(np.mean(ep))
    if epoch % 50 == 0 or epoch == cfg.epoch_diff-1:
        print(f'Epoch {epoch+1:>4} | time {time.time()-t0:.1f}s | loss {np.mean(ep):.6f}')
print('Training selesai.')

## Cell 9 — Inferensi & Imputasi

Model menghasilkan sampel imputasi pada posisi target. Median dari sampel dipakai sebagai
taksiran titik. Hasil ini adalah keluaran mentah model, sebelum survey rule engine.

In [ ]:
model.eval()
imputed_median = np.zeros_like(data)
samples_all = []
with torch.no_grad():
    idx=0
    for obs_d, obs_m, obs_tp, gt_m in test_loader:
        obs_d=obs_d.to(cfg.device); obs_m=obs_m.to(cfg.device)
        side_info = model.get_side_info(obs_tp.to(cfg.device), obs_m)
        s = model.impute(obs_d, obs_m, side_info, n_samples=cfg.n_samples)  # [B,n,K,L]
        med = s.median(dim=1).values                                       # [B,K,L]
        med = med.permute(0,2,1).cpu().numpy()                             # [B,L,K]
        b = med.shape[0]
        imputed_median[idx:idx+b] = med
        samples_all.append(s.permute(0,1,3,2).cpu().numpy())               # [B,n,L,K]
        idx += b
samples_all = np.concatenate(samples_all, axis=0)
print('Inferensi selesai. Bentuk imputasi:', imputed_median.shape)

## Cell 10 — Evaluasi Akurasi (MAE, RMSE, CRPS)

In [ ]:
# Denormalisasi ke skala asli untuk MAE/RMSE yang bermakna? -> evaluasi pd skala ternormalisasi (konsisten benchmark)
tgt = eval_target   # [N,L,K]
true_vals = data[tgt]
pred_vals = imputed_median[tgt]

mae  = np.mean(np.abs(pred_vals - true_vals))
rmse = np.sqrt(np.mean((pred_vals - true_vals)**2))

# CRPS pakai sampel penuh di posisi target
crps_list=[]
Nw,ns,Lw,Kw = samples_all.shape
ti = np.argwhere(tgt)
for (a,b,c) in ti:
    crps_list.append(ps.crps_ensemble(data[a,b,c], samples_all[a,:,b,c]))
crps = np.mean(crps_list) if crps_list else float('nan')

print('=== AKURASI (skala ternormalisasi) ===')
print(f'  MAE  = {mae:.4f}')
print(f'  RMSE = {rmse:.4f}')
print(f'  CRPS = {crps:.4f}')

## Cell 11 — Rekonstruksi Data Terimputasi (skala asli)

Nilai imputasi didenormalisasi kembali ke skala asli, lalu disusun ulang menjadi tabel
rumah tangga dengan kolom fitur semula. Nilai teramati dipertahankan; hanya posisi
artificial/natural missing yang diisi hasil model.

In [ ]:
# gabungkan: mulai dari data asli ternormalisasi, isi posisi target & natural missing dgn imputasi
filled = data.copy()
fill_pos = (cond_mask==0)   # semua posisi yang tak jadi kondisi (target + natural)
filled[fill_pos] = imputed_median[fill_pos]

# balik ke [n_ok, n_feat] lalu denormalisasi
flat = filled.reshape(-1, n_feat)[:n_rows]
flat_denorm = flat * stds + means

df_imp = df_support.iloc[:len(flat_denorm)].copy().reset_index(drop=True)
for k, cname in enumerate(FEAT_COLS):
    df_imp[cname] = flat_denorm[:, k]
print('Data terimputasi tersusun:', df_imp.shape)
df_imp.head()

## Cell 12 — Structural Violation Rate (SVR): Dua Skenario

- **Skenario A (tanpa survey rule engine):** keluaran imputasi apa adanya. Model mengisi
  posisi structural yang seharusnya kosong.
- **Skenario B (dengan survey rule engine):** posisi structural dikembalikan kosong via
  restore_structural.

SVR = proporsi posisi structural (yang seharusnya kosong) yang justru terisi.

In [ ]:
def is_filled(v):
    if pd.isna(v): return False
    if isinstance(v,str) and v.strip()=='': return False
    return True

CHECK_COLS = [col('R705'), col('R706'), col('R707')]

# hitung mask structural dari data asli (pakai R703_A + R705)
df_prep = sr.prepare_susenas_columns(df_imp)
valid_mask = sr.get_valid_mask(df_prep, sr.SUSENAS_SKIP_RULES)
structural = ~valid_mask
cols_all = list(df_prep.columns)

def svr(df_check):
    total, viol = 0, 0
    for v in CHECK_COLS:
        if v not in cols_all: continue
        j = cols_all.index(v)
        for i in range(len(df_check)):
            # posisi structural yang pada DATA ASLI kosong
            if structural[i,j] and not is_filled(df_support.iloc[i][v] if v in df_support.columns else np.nan):
                total += 1
                if is_filled(df_check.iloc[i][v]):
                    viol += 1
    return viol, total, (viol/total if total else 0.0)

# Skenario A: tanpa rule engine (df_imp apa adanya, model sudah isi structural)
vA, tA, svrA = svr(df_imp)

# Skenario B: dengan rule engine
df_ruled = sr.restore_structural(df_imp, df_prep, sr.SUSENAS_SKIP_RULES)
vB, tB, svrB = svr(df_ruled)

print('================ HASIL SVR ================')
print(f'[Skenario A] TANPA survey rule engine : {vA}/{tA} = {svrA:.2%} structural salah terisi')
print(f'[Skenario B] DENGAN survey rule engine : {vB}/{tB} = {svrB:.2%} structural salah terisi')

## Cell 13 — Simpan Output Terpisah (dengan vs tanpa rule engine)

In [ ]:
# Output tanpa rule engine
out_A = f'{WORK_DIR}/susenas_imputed_TANPA_rule.xlsx'
df_imp.to_excel(out_A, index=False)

# Output dengan rule engine
out_B = f'{WORK_DIR}/susenas_imputed_DENGAN_rule.xlsx'
df_ruled.drop(columns=[c for c in ['_r703_bekerja','_working'] if c in df_ruled.columns]).to_excel(out_B, index=False)

# Tabel ringkasan SVR + akurasi
ringkasan = pd.DataFrame({
    'Skenario': ['Tanpa Survey Rule', 'Dengan Survey Rule'],
    'Structural Salah Terisi': [vA, vB],
    'Total Structural': [tA, tB],
    'SVR (%)': [round(svrA*100,2), round(svrB*100,2)],
    'MAE': [round(mae,4), round(mae,4)],
    'RMSE': [round(rmse,4), round(rmse,4)],
    'CRPS': [round(crps,4), round(crps,4)],
})
out_R = f'{WORK_DIR}/susenas_ringkasan_SVR.xlsx'
ringkasan.to_excel(out_R, index=False)

print('File tersimpan:')
print(' -', out_A, '(imputasi TANPA rule engine)')
print(' -', out_B, '(imputasi DENGAN rule engine)')
print(' -', out_R, '(ringkasan SVR + akurasi)')
print()
print(ringkasan.to_string(index=False))